In [ ]:
import torch
import torch_geometric
import lightning as L
from rdkit.Chem import rdmolfiles
from torch_geometric.utils.smiles import from_rdmol
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool

import torch.nn.functional as F
from torch import nn
from torch import optim

import pandas as pd

from data_processing.common.ids import canonicalize_smiles
from pathlib import Path
import gc
import re
import shutil

from models.common.lightning_callbacks import EpochLossHistoryCallback
from models.pyg_pose.data import PoseAffinityDataset

Workflow:
- Create toy dataset.
    - This dataset should be pathologically easy. 
    - Maybe just like 10 graphs
- Test real model architecture on the toy dataset
- Train that model. 

# Create Toy Dataset
Loading a mini testing dataset of pytorch graphs to test the model training pipeline without lmdb pain

In [ ]:
VARIANT_SUFFIX_RE = re.compile(r"^(?P<ligand>.+)-(?P<variant_idx>\d+)$")

def _parse_variant_ligand(variant: str) -> str:
    """Strip the random suffix from `s_lp_Variant` and canonicalize the ligand."""
    variant = str(variant).strip()
    match = VARIANT_SUFFIX_RE.match(variant)
    if not match:
        raise ValueError(
            "Expected s_lp_Variant to look like '{smiles}-{int}', "
            f"got: {variant}"
        )
    return canonicalize_smiles(match.group("ligand"))

def _iter_ligand_mols(sdf_path: Path):
    """Yield `(mol_index, mol)` for valid ligand molecules in a docked SDF."""
    supplier = rdmolfiles.SDMolSupplier(str(sdf_path), removeHs=False)
    for mol_index, mol in enumerate(supplier):
        if mol is None:
            continue
        if not mol.HasProp("s_lp_Variant"):
            continue
        if not mol.HasProp("r_i_docking_score"):
            continue
        if not mol.HasProp("s_i_glide_gridfile"):
            continue
        yield mol_index, mol

def _read_sdf_rows(sdf_path: Path) -> list[dict[str, object]]:
    """Load docked SDF molecules with normalized metadata."""
    rows = []
    resolved_sdf = str(sdf_path.resolve())
    for mol_index, mol in _iter_ligand_mols(sdf_path):
        rows.append(
            {
                "mol_index": mol_index,
                "mol": mol,
                "ligand": _parse_variant_ligand(mol.GetProp("s_lp_Variant")),
                "grid": mol.GetProp("s_i_glide_gridfile").strip(),
                "glide_score": float(mol.GetProp("r_i_docking_score")),
                "source_sdf": resolved_sdf,
            }
        )
    return rows

def rdmol_to_pyg_with_pos(mol):
    """Convert an RDKit molecule into a PyG graph with 3D coordinates."""
    data = from_rdmol(mol)
    conf = mol.GetConformer()
    pos = conf.GetPositions()
    data.pos = torch.tensor(pos, dtype=torch.float)
    data.is_protein_atom = torch.zeros(data.num_nodes, dtype=torch.bool)
    return data

In [ ]:
rows = _read_sdf_rows(Path("tests/7BU7_P08588_docked/batch_1_docked.sdf"))
graph = rdmol_to_pyg_with_pos(rows[0]["mol"])
graph

In [ ]:
# prev = rows[0]["mol"].GetNumAtoms()
# for i in range(1000):
#     curr = rows[i]["mol"].GetNumAtoms()
#     if curr != prev:
#         print(i, curr)
#     prev = curr

# ok so we found that at i = 59 prev != curr. There's other examples but this is the first one. 
# our toy dataset will just have 2 graphs. i = 59 and i = 58. 
print(rows[58]["mol"].GetNumAtoms())
print(rows[59]["mol"].GetNumAtoms())

In [ ]:
g1 = rdmol_to_pyg_with_pos(rows[58]["mol"])
g2 = rdmol_to_pyg_with_pos(rows[59]["mol"])

import numpy as np
g1.y = torch.tensor([-10.0])
g2.y = torch.tensor([-1.0])

In [ ]:
print(  torch.all((g1.edge_index == g2.edge_index)),
        torch.all((g1.edge_attr == g2.edge_attr)),
        torch.all((g1.pos == g2.pos)),
        torch.all((g1.x == g2.x))
)


In [ ]:
# The training cells below now stream graphs from LMDB through PoseAffinityDataset
# instead of constructing DataLoader directly from this in-memory list.
data_list = [g1, g2]

# Stream Toy Dataset from LMDB

Create a tiny pose CSV with the same two scratch datapoints, write their PyG graphs to LMDB, then let `PoseAffinityDataset` attach labels and stream graph objects into the training loop.

In [ ]:
import lmdb
import pickle

pose_lmdb_dir = Path("tests/toy_lmdb")
toy_pose_csv = Path("tests/toy_pose_csv.csv")

# Keep the scratch notebook rerunnable while the real pipeline owns persistent LMDBs.
if pose_lmdb_dir.exists():
    shutil.rmtree(pose_lmdb_dir)
pose_lmdb_dir.mkdir(parents=True, exist_ok=True)
toy_pose_csv.parent.mkdir(parents=True, exist_ok=True)

pose_rows = pd.DataFrame(
    [
        {"pose_id": "toy_pose_58_atoms", "affinity": -10.0, "split": "train"},
        {"pose_id": "toy_pose_57_atoms", "affinity": -1.0, "split": "val"},
    ]
)
pose_rows.to_csv(toy_pose_csv, index=False)

pose_graphs = {
    "toy_pose_58_atoms": g1,
    "toy_pose_57_atoms": g2,
}

env = lmdb.open(
    str(pose_lmdb_dir),
    subdir=True,
    map_size=int(1e8),
    metasync=False,
    sync=False,
    lock=False,
    readahead=False,
    meminit=False,
)
try:
    with env.begin(write=True) as txn:
        for pose_id, graph in pose_graphs.items():
            graph.pose_id = pose_id
            txn.put(pose_id.encode("utf-8"), pickle.dumps(graph), overwrite=False)
finally:
    env.close()
    del env
    gc.collect()

toy_dataset = PoseAffinityDataset(toy_pose_csv, pose_lmdb_dir)
train_indices = toy_dataset.df.index[toy_dataset.df["split"] == "train"].tolist()
val_indices = toy_dataset.df.index[toy_dataset.df["split"] == "val"].tolist()

train_dataset = torch.utils.data.Subset(toy_dataset, train_indices)
val_dataset = torch.utils.data.Subset(toy_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)

print(pose_rows)
print(train_dataset[0])
print(val_dataset[0])

# Lightning GNN

In [ ]:
# define the LightningModule
class LitPoseGNN(L.LightningModule):
    def __init__(self, num_atom_features, hidden_dim):
        super().__init__()
        num_atom_features = num_atom_features + 3 # atom features, coordinates. 
        self.conv1 = GCNConv(in_channels=num_atom_features,out_channels=hidden_dim)
        self.conv2 = GCNConv(in_channels=hidden_dim,out_channels=hidden_dim)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, batch):
        x, coords, edge_index, batch_idx = batch.x.float(), batch.pos, batch.edge_index, batch.batch
        # coords = torch.zeros_like(coords) # no coordinates for now. coords negative control
        x = torch.cat([x, coords], dim=1)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch_idx)
        x = self.regressor(x)
        return x

    def training_step(self, batch):
        # training_step defines the train loop.
        # it is independent of forward
        x = self.forward(batch)
        y = batch.y.view_as(x).float()
        loss = F.mse_loss(x, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch):
        x = self.forward(batch)
        y = batch.y.view_as(x).float()
        loss = F.mse_loss(x, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [ ]:
num_atom_features = train_loader.dataset[0].x.shape[1]
model = LitPoseGNN(num_atom_features=num_atom_features, hidden_dim=128)


checkpoint_callback = L.pytorch.callbacks.ModelCheckpoint(
    save_top_k=1,
    dirpath=Path("runs/pyg_pose_scratch/checkpoints"),
    filename="best",
    monitor="val_loss",
    mode="min",
    save_last=True,
    enable_version_counter=False
)

loss_history = EpochLossHistoryCallback()

trainer = L.Trainer(    logger=False,
                        enable_checkpointing=True,
                        enable_progress_bar=True,
                        accelerator="auto",
                        max_epochs=200,
                        default_root_dir=Path("runs/pyg_pose_scratch"),
                        callbacks=[
                            checkpoint_callback,
                            loss_history.bind_lightning_callback()
                        ]
                    )

trainer.fit(        model=model, 
                    train_dataloaders=train_loader,
                    val_dataloaders=val_loader)

In [ ]:
model(g2)

In [ ]:
g1, g2

In [ ]:
# load checkpoint
checkpoint = "./runs/pyg_pose_scratch/checkpoints/last.ckpt"
device = torch.device('cpu')
poseGNN = LitPoseGNN.load_from_checkpoint(checkpoint, num_atom_features=num_atom_features, hidden_dim=128).to(device)

poseGNN(g2)